In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, classification_report

# ---------------------------------------------------------
# 1. Load Dataset & Preprocessing
# ---------------------------------------------------------
# Read diabetes dataset
df = pd.read_csv('diabetes.csv')

# Split features (X) and target (y)
X = df.drop(columns=['Outcome'])
y = df['Outcome']

# Split train and test sets (80% train, 20% test) with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------------------------------------------------
# 2. Hypertuning 'k' using GridSearchCV
# ---------------------------------------------------------
# Create KNN classifier
knn = KNeighborsClassifier()

# Define grid search parameter range for k (1 to 25)
param_grid = {'n_neighbors': np.arange(1, 25)}

# Use GridSearchCV with 5-fold cross-validation
knn_gscv = GridSearchCV(knn, param_grid, cv=5)
knn_gscv.fit(X_train_scaled, y_train)

best_k = knn_gscv.best_params_['n_neighbors']
print(f"Optimal number of neighbors (k): {best_k}")
print(f"Best Cross-Validation Score: {knn_gscv.best_score_ * 100:.2f}%")

# ---------------------------------------------------------
# 3. Model Training & Evaluation with Optimal k
# ---------------------------------------------------------
optimal_knn = KNeighborsClassifier(n_neighbors=best_k)
optimal_knn.fit(X_train_scaled, y_train)
y_pred = optimal_knn.predict(X_test_scaled)

# Metrics calculation
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
error_rate = 1 - acc

print("\n=== KNN MODEL EVALUATION ===")
print(f"Accuracy: {acc * 100:.2f}%")
print(f"Error Rate: {error_rate * 100:.2f}%")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Optimal number of neighbors (k): 19
Best Cross-Validation Score: 76.05%

=== KNN MODEL EVALUATION ===
Accuracy: 74.68%
Error Rate: 25.32%
Precision: 0.7027
Recall: 0.4815

Confusion Matrix:
 [[89 11]
 [28 26]]

Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.89      0.82       100
           1       0.70      0.48      0.57        54

    accuracy                           0.75       154
   macro avg       0.73      0.69      0.70       154
weighted avg       0.74      0.75      0.73       154

